# EMA + RSI

EMA Crossover + RSI Filter \
A momentum strategy that trades in the direction of the cross: fast EMA(9) over slow EMA(21) \
It uses ATR(14) for a dynamic volatility-adaptive trailing stop. \
Filters false signals with RSI(14) to cut whipsaws in ranging markets. \
Works across timeframes; well suited to scalping and intraday.

__How EMA+RSI Algorithm Determines Entry/Exit:__
- Fast EMA (9) / Slow EMA (21) – standard for crypto.
- Long Entry: Fast EMA crosses above Slow EMA AND RSI(14) < 70 (not overbought).
- Short Entry: Fast EMA crosses below Slow EMA AND RSI(14) > 30 (not oversold).
- Exit: Reverse crossover (signal flip) OR price hits the ATR-based trailing stop.
- The RSI filter reduces whipsaws in ranging markets.

__The RSI filter__ on ema / ema_inv is an optional, configurable filter:
- filter off entirely: StrategyConfig(rsi_filter=False)
- custom bounds: StrategyConfig(rsi_bullish=65.0, rsi_bearish=35.0)

## Configuration: automatic

In [1]:
# --- path bootstrap: make 'engine' importable from any CWD ---
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / "engine" / "__init__.py").exists():
    if root == root.parent: raise RuntimeError("project root not found")
    root = root.parent
if str(root) not in sys.path: sys.path.insert(0, str(root))

In [2]:
from engine.backtester import Backtester
from engine.trade_configurator import ACTIVE_TRADE
from engine.core import Signal, SignalAction
from engine.strategy_configurator import StrategyConfig
from engine.visualization import build_chart

In [ ]:
# Automatic config: canonical handles from the three configurators (project-wide defaults).
# Override any of them in the manual chapter below; leave its dicts empty to stay automatic.
import dataclasses
from engine.data_configurator import ACTIVE, load_data, save_result
from engine.strategy_configurator import StrategyConfig, EXIT_PRESETS
from engine.trade_configurator import ACTIVE_TRADE

DATA_CONFIG     = ACTIVE              # engine/data_configurator.py     (DataSpec)
STRATEGY_CONFIG = StrategyConfig()    # engine/strategy_configurator.py (signal knobs)
TRADING_CONFIG  = ACTIVE_TRADE        # engine/trade_configurator.py    (TradingConfig)
EXIT_POLICY     = None                # None -> each strategy's assigned default (exit_policy_for)

## Configuration: manual

Per-notebook overrides on top of the automatic config above. Each cell applies
`dataclasses.replace` to one handle; **leave a dict empty (or `EXIT_POLICY = None`)
to keep that dimension automatic**. Everything below this chapter uses only
`df`, `SYMBOL`, `INTERVAL`, `STRATEGY_CONFIG`, `EXIT_POLICY`, `TRADING_CONFIG`.

In [ ]:
# manual override — DATA. Fill the dict to override; empty = automatic (ACTIVE).
DATA_OVERRIDES = {}          # e.g. {"symbol": "ETHUSDT", "interval": "60", "num_candles": 1500}
DATA_CONFIG = dataclasses.replace(DATA_CONFIG, **DATA_OVERRIDES)

In [ ]:
# manual override — STRATEGY signal knobs. Empty = automatic StrategyConfig().
STRATEGY_OVERRIDES = {}      # e.g. {"ema_fast": 12, "ema_slow": 26, "rsi_filter": False}
STRATEGY_CONFIG = dataclasses.replace(STRATEGY_CONFIG, **STRATEGY_OVERRIDES)

In [ ]:
# manual override — EXIT policy. None = each strategy's assigned default.
# Preset: EXIT_POLICY = EXIT_PRESETS["fixed_2pct_rr3"]()
# Custom: from engine.exits import CompositeExit, AtrStop, RrTarget
#         EXIT_POLICY = CompositeExit(AtrStop(1.5), RrTarget(2.0))
EXIT_POLICY = None

In [ ]:
# manual override — TRADE config (costs / sizing / leverage / direction). Empty = ACTIVE_TRADE.
# For direction: from engine.trade_configurator import TradeDirection
TRADE_OVERRIDES = {}         # e.g. {"leverage": 2.0, "direction": TradeDirection.LONG}
TRADING_CONFIG = dataclasses.replace(TRADING_CONFIG, **TRADE_OVERRIDES)

In [ ]:
# Resolve the final inputs the rest of the notebook uses. (Runs after overrides.)
df = load_data(DATA_CONFIG)
SYMBOL, INTERVAL = DATA_CONFIG.symbol, DATA_CONFIG.interval

## EMA + RSI

In [5]:
# Import EMA + RSI strategy
from engine.strategies import EMACrossoverStrategy

In [ ]:
# Backtest EMA + RSI strategy
strategy = EMACrossoverStrategy(STRATEGY_CONFIG, exit_policy=EXIT_POLICY)
bt = Backtester(strategy, symbol=SYMBOL, trading_config=TRADING_CONFIG)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

# Save metrics (JSON) + trade log (CSV) under data/results/<dataset signature>/
save_result(result, DATA_CONFIG)

In [ ]:
# EMA + RSI strategy chart
prepared = strategy.prepare(df)
signals = []
for t in result.trades:
    if t.entry_ts:
        signals.append(Signal(t.entry_ts, SignalAction.ENTRY, t.direction, t.entry_price))
    if t.exit_ts:
        signals.append(Signal(t.exit_ts, SignalAction.EXIT, t.direction, t.exit_price))

build_chart(
    prepared, signals,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
).show()

# Inverse EMA + RSI

Inverse EMA Crossover + RSI Filter \
Mean-reversion counterpart to EMA+RSI: fades the cross instead of riding it. \
Uses ATR(14) for dynamic trailing stops (adapts to volatility) and RSI(14) filter. \
Bets that EMA crosses mark momentum exhaustion, not continuation.

__How Inverse EMA+RSI Algorithm Determines Entry/Exit:__
- Fast EMA (9) / Slow EMA (21) — standard for crypto.
- Short Entry: Fast EMA crosses above Slow EMA AND RSI(14) > 30 (not oversold) — fading the bullish cross.
- Long Entry:  Fast EMA crosses below Slow EMA AND RSI(14) < 70 (not overbought) — fading the bearish cross.
- Exit: Opposite crossover (signal flip) OR price hits ATR-based trailing stop.
- Works best in range-bound / mean-reverting regimes; likely underperforms in strong trends.


In [ ]:
# Import Inverse EMA + RSI strategy
from engine.strategies import InverseEMACrossoverStrategy

In [ ]:
# Backtest Inverse EMA + RSI strategy
strategy = InverseEMACrossoverStrategy(STRATEGY_CONFIG, exit_policy=EXIT_POLICY)
bt = Backtester(strategy, symbol=SYMBOL, trading_config=TRADING_CONFIG)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

# Save metrics (JSON) + trade log (CSV) under data/results/<dataset signature>/
save_result(result, DATA_CONFIG)

In [ ]:
# Inverse EMA + RSI strategy chart
prepared = strategy.prepare(df)
signals = []
for t in result.trades:
    if t.entry_ts:
        signals.append(Signal(t.entry_ts, SignalAction.ENTRY, t.direction, t.entry_price))
    if t.exit_ts:
        signals.append(Signal(t.exit_ts, SignalAction.EXIT, t.direction, t.exit_price))

build_chart(
    prepared, signals,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
).show()